In [1]:
import pickle, statistics, os, json, re, httpx, getpass, fnmatch
from collections import Counter
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from helpers import score_answer, parse_answers, score_run

CATEGORIES = ['structure', 'deps', 'imports', 'classes', 'patterns', 'testing', 'config', 'docs']

def load_results(label):
    with open(f'results/{label}.pkl', 'rb') as f:
        return pickle.load(f)

exps = {label: load_results(label) for label in
        ['baseline', 'plan', 'cot', 'thinking_high']}
cons = load_results('consistency')
print(f'loaded {len(exps)} experiments + consistency ({len(cons)} runs)')

loaded 7 experiments + consistency (5 runs)


In [2]:
print(f'{"Experiment":<16} {"Acc":<8} {"Files":<7} {"Loops":<7} {"Halluc":<7}')
print('-' * 45)
for label, runs in exps.items():
    ok = [r for r in runs if not r.get('crashed')]
    if not ok:
        print(f'{label:<16} ALL CRASHED')
        continue
    acc = statistics.mean([r['accuracy'] for r in ok])
    files = statistics.mean([r['files_read'] for r in ok])
    loops = statistics.mean([r['loops'] for r in ok])
    halluc = statistics.mean([r['hallucinations'] for r in ok])
    print(f'{label:<16} {acc:.0%}     {files:.0f}     {loops:.0f}      {halluc:.0f}')

Experiment       Acc      Files   Loops   Halluc 
---------------------------------------------
baseline         79%     5     11      9
plan             79%     4     10      10
reflect          79%     10     17      9
cot              89%     20     24      5
think            59%     21     9      4
thinking_low     84%     21     19      7
thinking_high    89%     31     20      5


In [3]:
obs_ids = [7, 8, 10, 11, 13, 21, 24, 25, 26, 27, 28, 30, 32, 33, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 48, 50]
comp_ids = [1, 2, 5, 16, 18, 19, 22, 31, 35, 43, 47]

print(f'{"Experiment":<16} {"Observation":<14} {"Computation":<14} {"Gap"}')
print('-' * 50)
for label in ['baseline', 'plan', 'cot', 'thinking_high']:
    ok = [r for r in exps[label] if not r.get('crashed')]
    obs = statistics.mean([statistics.mean([r['scores'][q]['score'] for q in obs_ids]) for r in ok])
    comp = statistics.mean([statistics.mean([r['scores'][q]['score'] for q in comp_ids]) for r in ok])
    print(f'{label:<16} {obs:<14.0%} {comp:<14.0%} {obs-comp:+.0%}')

Experiment       Observation    Computation    Gap
--------------------------------------------------
baseline         88%            55%            +34%
plan             88%            48%            +40%
cot              97%            70%            +28%
thinking_high    96%            76%            +20%


In [4]:
counting_qs = [1, 2, 5, 16, 18, 19, 22, 31, 35, 43, 47]
ok = [r for r in cons if not r.get('crashed')]

print('Counting questions — 5 runs, same repo, same data in context')
print(f'{"Q":<5} {"Truth":>6}  {"Answers":<40} {"Correct":<10} {"Confidence"}')
print('-' * 85)
for qid in counting_qs:
    preds = [str(r['scores'][qid]['predicted'])[:7] for r in ok]
    truth = ok[0]['scores'][qid]['truth']
    correct = sum(1 for r in ok if r['scores'][qid]['score'] > 0)
    confs = [r.get('confidences', {}).get(qid) for r in ok]
    conf_str = ', '.join(f'{c:.0%}' if c else '?' for c in confs)
    print(f'Q{qid:<4} {str(truth):>6}  {str(preds):<40} {correct}/5       {conf_str}')

Counting questions — 5 runs, same repo, same data in context
Q      Truth  Answers                                  Correct    Confidence
-------------------------------------------------------------------------------------
Q1        62  ['77', '45', '19', '17', '65']           1/5       100%, 100%, 100%, 100%, 90%
Q2        23  ['33', '33', '29', '35', '23']           1/5       100%, 100%, 100%, 100%, 100%
Q5        20  ['20', '20', '19', '20', '21']           5/5       100%, 100%, 100%, 100%, 100%
Q16       67  ['62', '67', '53', '66', '56']           3/5       100%, 100%, 100%, 100%, 100%
Q18       11  ['8', '17', '13', '17', '17']            0/5       100%, 100%, 100%, 100%, 100%
Q19       19  ['15', '13', '15', '9', '12']            0/5       100%, 100%, 100%, 100%, 100%
Q22       11  ['11', '10', '10', '10', '10']           5/5       100%, 100%, 100%, 100%, 100%
Q31      372  ['300', '480', '200', '240', '250']      0/5       100%, 100%, 100%, 100%, 100%
Q35       80  ['53', '42'

In [5]:
# did it actually call get_repo_tree?
print('Q1: get_repo_tree called?')
for r in ok:
    tree_called = any('get_repo_tree' in c['name'] for s in r['stats'] for c in s['calls'])
    q1 = r['scores'][1]['predicted']
    right = r['scores'][1]['score'] > 0
    print(f'  Run {r["run"]}: tree={"YES" if tree_called else "NO"}  Q1={q1}  correct={right}')
# all YES. the data was there. the model can't count it.

Q1: get_repo_tree called?
  Run 1: tree=YES  Q1=77  correct=False
  Run 2: tree=YES  Q1=45  correct=False
  Run 3: tree=YES  Q1=19  correct=False
  Run 4: tree=YES  Q1=17  correct=False
  Run 5: tree=YES  Q1=65  correct=True


In [6]:
print(f'{"Experiment":<16} {"Files":<8} {"Accuracy"}')
print('-' * 35)
for label, runs in exps.items():
    ok = [r for r in runs if not r.get('crashed')]
    if not ok:
        continue
    files = statistics.mean([r['files_read'] for r in ok])
    acc = statistics.mean([r['accuracy'] for r in ok])
    print(f'{label:<16} {files:<8.0f} {acc:.0%}')

Experiment       Files    Accuracy
-----------------------------------
baseline         5        79%
plan             4        79%
reflect          10       79%
cot              20       89%
think            21       59%
thinking_low     21       84%
thinking_high    31       89%


In [7]:
print(f'{"Experiment":<16} {"Conf(right)":<14} {"Conf(wrong)":<14} {"High-conf wrong"}')
print('-' * 60)
for label, runs in exps.items():
    ok = [r for r in runs if not r.get('crashed')]
    pairs = []
    for r in ok:
        confs = r.get('confidences', {})
        for qid, sd in r['scores'].items():
            c = confs.get(qid)
            if c is not None:
                pairs.append((c, sd['score']))
    if not pairs:
        continue
    right_c = [c for c, s in pairs if s > 0]
    wrong_c = [c for c, s in pairs if s == 0]
    hc = sum(1 for c, s in pairs if c >= 0.8 and s == 0)
    print(f'{label:<16} {statistics.mean(right_c):<14.2f} {statistics.mean(wrong_c):<14.2f} {hc}')

Experiment       Conf(right)    Conf(wrong)    High-conf wrong
------------------------------------------------------------
baseline         0.96           0.93           27
plan             0.99           0.95           28
reflect          0.99           0.96           24
cot              1.00           0.99           15
think            1.00           0.91           10
thinking_low     1.00           1.00           22
thinking_high    1.00           0.93           14


---

## The fix

Fetch tools get the data. The model can't count what comes back. Add tools that count.

In [8]:
def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')

_set_env('GOOGLE_API_KEY')
_set_env('GITHUB_TOKEN')

GITHUB_HEADERS = {'Authorization': f'token {os.environ["GITHUB_TOKEN"]}'}

def _gh(endpoint):
    r = httpx.get(f'https://api.github.com{endpoint}', headers=GITHUB_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview')

In [9]:
# fetch tools

@tool
def get_repo_info(owner: str, repo: str) -> str:
    """Repo metadata."""
    d = _gh(f'/repos/{owner}/{repo}')
    return json.dumps({k: d.get(k) for k in
        ['full_name', 'description', 'language', 'stargazers_count', 'forks_count',
         'default_branch', 'has_wiki', 'topics']}, indent=2)

@tool
def get_repo_tree(owner: str, repo: str, branch: str = 'main') -> str:
    """Full file tree."""
    d = _gh(f'/repos/{owner}/{repo}/git/trees/{branch}?recursive=1')
    entries = d.get('tree', [])
    lines = [f'{e["type"]:4} {e["path"]}' for e in entries[:500]]
    return '\n'.join(lines)

@tool
def get_file_content(owner: str, repo: str, path: str) -> str:
    """Single file content."""
    r = httpx.get(f'https://api.github.com/repos/{owner}/{repo}/contents/{path}',
                  headers={**GITHUB_HEADERS, 'Accept': 'application/vnd.github.raw+json'}, timeout=15)
    r.raise_for_status()
    lines = r.text.split('\n')
    if len(lines) > 300:
        return '\n'.join(lines[:300]) + f'\n\n[TRUNCATED]'
    return r.text

@tool
def get_repo_languages(owner: str, repo: str) -> str:
    """Language breakdown."""
    return json.dumps(_gh(f'/repos/{owner}/{repo}/languages'))

@tool
def think(thought: str) -> str:
    """Scratchpad."""
    return 'Thought noted.'

In [10]:
# computation tools
import fnmatch

@tool
def count_files(owner: str, repo: str, pattern: str = '*.py') -> str:
    """Count files matching a glob."""
    d = _gh(f'/repos/{owner}/{repo}/git/trees/main?recursive=1')
    matches = [e for e in d.get('tree', []) if e['type'] == 'blob' and fnmatch.fnmatch(e['path'], pattern)]
    return f'{len(matches)} files match {pattern}'

@tool
def count_pattern(owner: str, repo: str, path: str, pattern: str) -> str:
    """Count regex matches. Handles globs."""
    if '*' in path or '?' in path:
        # glob — count across matching files
        d = _gh(f'/repos/{owner}/{repo}/git/trees/main?recursive=1')
        files = [e['path'] for e in d.get('tree', []) if e['type'] == 'blob' and fnmatch.fnmatch(e['path'], path)]
        total = 0
        per_file = []
        for fpath in files:
            try:
                r = httpx.get(f'https://api.github.com/repos/{owner}/{repo}/contents/{fpath}',
                              headers={**GITHUB_HEADERS, 'Accept': 'application/vnd.github.raw+json'}, timeout=15)
                r.raise_for_status()
                count = len(re.findall(pattern, r.text, re.MULTILINE))
                total += count
                per_file.append(f'{fpath}: {count}')
            except Exception:
                per_file.append(f'{fpath}: ERROR')
        return f'{total} total matches for `{pattern}` across {len(files)} files\n' + '\n'.join(per_file)
    else:
        # exact file
        r = httpx.get(f'https://api.github.com/repos/{owner}/{repo}/contents/{path}',
                      headers={**GITHUB_HEADERS, 'Accept': 'application/vnd.github.raw+json'}, timeout=15)
        r.raise_for_status()
        matches = re.findall(pattern, r.text, re.MULTILINE)
        return f'{len(matches)} matches for `{pattern}` in {path}'

@tool
def count_dirs(owner: str, repo: str) -> str:
    """Count directories."""
    d = _gh(f'/repos/{owner}/{repo}/git/trees/main?recursive=1')
    dirs = [e for e in d.get('tree', []) if e['type'] == 'tree']
    return f'{len(dirs)} directories'

# quick test
print(count_files.invoke({'owner': 'pallets', 'repo': 'click', 'pattern': '*.py'}))
print(count_pattern.invoke({'owner': 'pallets', 'repo': 'click', 'path': 'src/click/core.py', 'pattern': r'^class '}))
print(count_dirs.invoke({'owner': 'pallets', 'repo': 'click'}))


62 files match *.py
11 matches for `^class ` in src/click/core.py
23 directories


In [11]:
# counting questions only
GROUND_TRUTH_FULL = [
    {'id': 1, 'cat': 'structure', 'q': 'How many Python files (.py) are in this repo?', 'a': 62, 'type': 'int'},
    {'id': 2, 'cat': 'structure', 'q': 'How many directories are in this repo?', 'a': 23, 'type': 'int'},
    {'id': 5, 'cat': 'structure', 'q': 'How many test files (files starting with test_) exist?', 'a': 20, 'type': 'int'},
    {'id': 16, 'cat': 'imports', 'q': 'How many symbols does __init__.py export (count the from .xxx import lines)?', 'a': 67, 'type': 'int'},
    {'id': 18, 'cat': 'classes', 'q': 'How many class definitions are in src/click/core.py?', 'a': 11, 'type': 'int'},
    {'id': 19, 'cat': 'classes', 'q': 'How many class definitions are in src/click/types.py?', 'a': 19, 'type': 'int'},
    {'id': 22, 'cat': 'classes', 'q': 'How many class definitions are in src/click/exceptions.py?', 'a': 11, 'type': 'int'},
    {'id': 31, 'cat': 'testing', 'q': 'How many test functions (def test_) exist across all test files?', 'a': 372, 'type': 'int'},
    {'id': 35, 'cat': 'testing', 'q': 'How many test functions are in test_options.py?', 'a': 80, 'type': 'int'},
    {'id': 43, 'cat': 'config', 'q': 'How many GitHub Actions workflow files are there?', 'a': 5, 'type': 'int'},
    {'id': 47, 'cat': 'docs', 'q': 'How many lines is the README.md?', 'a': 62, 'type': 'int'},
]

QUESTIONS_TEXT = '\n'.join(
    f'{q["id"]}. {q["q"]} (answer type: {q["type"]})' for q in GROUND_TRUTH_FULL
)
print(f'{len(GROUND_TRUTH_FULL)} counting questions')

11 counting questions


In [12]:
run_stats = []

def build_graph(tools_list, sys_msg, model=None):
    _llm = model or llm
    llm_with_tools = _llm.bind_tools(tools_list)

    def llm_node(state: MessagesState):
        response = llm_with_tools.invoke([sys_msg] + state['messages'])
        loop_num = len(run_stats) + 1
        calls = [{'name': c['name'], 'args': {k: str(v)[:80] for k, v in c['args'].items()}} for c in response.tool_calls]
        if calls:
            print(f'    loop {loop_num}: {", ".join(c["name"] for c in calls)}', flush=True)
        else:
            print(f'    loop {loop_num}: generating answer', flush=True)
        run_stats.append({
            'loop': loop_num,
            'input_tokens': response.usage_metadata.get('input_tokens', 0) if response.usage_metadata else 0,
            'output_tokens': response.usage_metadata.get('output_tokens', 0) if response.usage_metadata else 0,
            'tool_call': bool(response.tool_calls),
            'calls': calls,
        })
        return {'messages': [response]}

    builder = StateGraph(MessagesState)
    builder.add_node('llm', llm_node)
    builder.add_node('tools', ToolNode(tools_list))
    builder.add_edge(START, 'llm')
    builder.add_conditional_edges('llm', tools_condition)
    builder.add_edge('tools', 'llm')
    return builder.compile()

In [13]:
# without computation tools
prompt_count = SystemMessage(content=(
    'You are a code analyzer. Answer these counting questions about pallets/click.\n'
    'Be exact. Count carefully.\n\n'
    'Output format: Q<id>: [confidence] <number>\n\n'
    f'QUESTIONS:\n{QUESTIONS_TEXT}'
))

fetch_tools = [get_repo_info, get_repo_tree, get_file_content, get_repo_languages, think]
graph_fetch = build_graph(fetch_tools, prompt_count)

FIX_RESULTS_PATH = 'results/fix_comparison.pkl'
if os.path.exists(FIX_RESULTS_PATH):
    with open(FIX_RESULTS_PATH, 'rb') as f:
        fix_data = pickle.load(f)
    print(f'loaded cached fix comparison')
else:
    fix_data = {'without': [], 'with': []}

    # 3 runs without computation tools
    print('Without computation tools:')
    for i in range(3):
        run_stats.clear()
        try:
            result = graph_fetch.invoke(
                {'messages': [HumanMessage(content='Answer these counting questions about pallets/click.')]},
                config={'recursion_limit': 120}, version='v2')
            raw = result.value['messages'][-1].content
            text = '\n'.join(p['text'] for p in raw if isinstance(p, dict) and 'text' in p) if isinstance(raw, list) else str(raw)
            answers, confidences = parse_answers(text)
            scores = score_run(answers, GROUND_TRUTH_FULL)
            acc = sum(s['score'] for s in scores.values()) / len(scores)
            fix_data['without'].append({'answers': answers, 'scores': scores, 'accuracy': acc, 'confidences': confidences})
            print(f'  Run {i+1}: acc={acc:.0%}  answers={[(q, answers.get(q)) for q in [1, 2, 18, 31]]}')
        except Exception as e:
            fix_data['without'].append({'crashed': True, 'error': str(e)})
            print(f'  Run {i+1}: CRASHED -- {e}')

Without computation tools:
    loop 1: get_repo_info
    loop 2: get_repo_tree
    loop 3: get_file_content
    loop 4: get_file_content
    loop 5: get_file_content
    loop 6: get_file_content
    loop 7: get_file_content
    loop 8: get_file_content
    loop 9: get_repo_tree
    loop 10: generating answer
  Run 1: acc=45%  answers=[(1, '77'), (2, '35'), (18, '9'), (31, '337')]
    loop 1: get_repo_info
    loop 2: get_repo_tree
    loop 3: think
    loop 4: get_file_content
    loop 5: get_file_content
    loop 6: get_file_content
    loop 7: get_file_content
    loop 8: get_file_content
    loop 9: get_file_content
    loop 10: get_repo_tree
    loop 11: get_file_content
    loop 12: generating answer
  Run 2: acc=45%  answers=[(1, '62'), (2, '24'), (18, '8'), (31, '212')]
    loop 1: get_repo_info
    loop 2: get_repo_tree
    loop 3: think
    loop 4: get_file_content
    loop 5: get_file_content
    loop 6: get_file_content
    loop 7: get_file_content
    loop 8: get_file_conte

In [14]:
# with computation tools
if not fix_data['with']:
    compute_tools = [get_repo_info, get_repo_tree, get_file_content, get_repo_languages, think,
                     count_files, count_pattern, count_dirs]
    graph_compute = build_graph(compute_tools, prompt_count)

    print('With computation tools:')
    for i in range(3):
        run_stats.clear()
        try:
            result = graph_compute.invoke(
                {'messages': [HumanMessage(content='Answer these counting questions about pallets/click.')]},
                config={'recursion_limit': 120}, version='v2')
            raw = result.value['messages'][-1].content
            text = '\n'.join(p['text'] for p in raw if isinstance(p, dict) and 'text' in p) if isinstance(raw, list) else str(raw)
            answers, confidences = parse_answers(text)
            scores = score_run(answers, GROUND_TRUTH_FULL)
            acc = sum(s['score'] for s in scores.values()) / len(scores)
            fix_data['with'].append({'answers': answers, 'scores': scores, 'accuracy': acc, 'confidences': confidences})
            print(f'  Run {i+1}: acc={acc:.0%}  answers={[(q, answers.get(q)) for q in [1, 2, 18, 31]]}')
        except Exception as e:
            fix_data['with'].append({'crashed': True, 'error': str(e)})
            print(f'  Run {i+1}: CRASHED -- {e}')

    with open(FIX_RESULTS_PATH, 'wb') as f:
        pickle.dump(fix_data, f)
    print(f'saved to {FIX_RESULTS_PATH}')

With computation tools:
    loop 1: get_repo_info
    loop 2: count_files
    loop 3: count_dirs
    loop 4: count_files
    loop 5: get_file_content
    loop 6: count_pattern
    loop 7: count_pattern
    loop 8: count_pattern
    loop 9: count_pattern
    loop 10: count_pattern
    loop 11: count_pattern
    loop 12: count_files
    loop 13: get_repo_tree
    loop 14: count_files
    loop 15: get_file_content
    loop 16: count_pattern
    loop 17: count_pattern
    loop 18: count_pattern
    loop 19: count_pattern
    loop 20: count_pattern
    loop 21: count_pattern
    loop 22: count_pattern
    loop 23: generating answer
  Run 1: acc=100%  answers=[(1, '62'), (2, '23'), (18, '11'), (31, '381')]
    loop 1: get_repo_info
    loop 2: count_files
    loop 3: count_dirs
    loop 4: count_files
    loop 5: get_file_content
    loop 6: count_pattern
    loop 7: count_pattern
    loop 8: count_pattern
    loop 9: count_pattern
    loop 10: count_pattern
    loop 11: count_files
    loop

In [15]:
# before vs after
print(f'{"Question":<50} {"Without":<20} {"With":<20} {"Truth"}')
print('-' * 100)

without_ok = [r for r in fix_data['without'] if not r.get('crashed')]
with_ok = [r for r in fix_data['with'] if not r.get('crashed')]

for gt in GROUND_TRUTH_FULL:
    qid = gt['id']
    w_answers = [str(r['answers'].get(qid, '?'))[:8] for r in without_ok]
    c_answers = [str(r['answers'].get(qid, '?'))[:8] for r in with_ok]
    w_correct = sum(1 for r in without_ok if r['scores'][qid]['score'] > 0)
    c_correct = sum(1 for r in with_ok if r['scores'][qid]['score'] > 0)
    print(f'Q{qid}: {gt["q"][:45]:<50} {str(w_answers):<20} {str(c_answers):<20} {gt["a"]}')

w_acc = statistics.mean([r['accuracy'] for r in without_ok]) if without_ok else 0
c_acc = statistics.mean([r['accuracy'] for r in with_ok]) if with_ok else 0
print(f'\nOverall counting accuracy: {w_acc:.0%} -> {c_acc:.0%}')

Question                                           Without              With                 Truth
----------------------------------------------------------------------------------------------------
Q1: How many Python files (.py) are in this repo?      ['77', '62', '62']   ['62', '62', '62']   62
Q2: How many directories are in this repo?             ['35', '24', '23']   ['23', '23', '23']   23
Q5: How many test files (files starting with test      ['21', '20', '20']   ['20', '20', '20']   20
Q16: How many symbols does __init__.py export (cou      ['71', '58', '62']   ['62', '62', '66']   67
Q18: How many class definitions are in src/click/c      ['9', '8', '9']      ['11', '11', '11']   11
Q19: How many class definitions are in src/click/t      ['9', '7', '12']     ['19', '19', '19']   19
Q22: How many class definitions are in src/click/e      ['12', '10', '13']   ['11', '11', '11']   11
Q31: How many test functions (def test_) exist acr      ['337', '212', '250'] ['381', '379', '37